# Multi-Turn Conversations — Claude Managed Agents (CLI)

The same multi-turn flow as [`02-multi-turn/`](../02-multi-turn/), driven from the terminal with Anthropic's **`ant`** CLI instead of the Python SDK.

The headline idea is identical to the SDK version: **create one session and reuse it across turns.** Conversation history lives server-side, so each new message automatically sees everything that came before — you never resend prior turns. The only new mechanics here are shell-flavored:

- Agents and environments are defined as version-controlled `*.yaml` (control plane).
- Each turn repeats the **stream-first** dance — open the stream *before* sending — wrapped in a reusable `chat.sh` helper so the turn cells stay one line each.

> Prerequisite: the `ant` CLI installed and authenticated. See [`01-basics-cli/`](../01-basics-cli/) for install/auth details.

## 1. Setup

Same as module 01. `ant` is a separate process from this notebook, so the cell below loads the root `.env` into `os.environ` (every `!ant` call inherits the key) and clears Jupyter's `FORCE_COLOR`, which would otherwise make `ant` wrap its JSON/YAML in ANSI color codes and break parsing.

> Prerequisite: the `ant` CLI installed and authenticated — see [`01-basics-cli/`](../01-basics-cli/) for install/auth details.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# `ant` is a separate process from this notebook, so two bits of setup:
#  - it doesn't read the root .env — load the API key into os.environ, which
#    every `!ant` call below inherits;
#  - Jupyter sets FORCE_COLOR=1, which makes ant wrap its JSON/YAML output in
#    ANSI color codes and break parsing — clear it so the output stays plain.
load_dotenv(find_dotenv())
os.environ.pop("FORCE_COLOR", None)
os.environ.pop("CLICOLOR_FORCE", None)
os.environ["NO_COLOR"] = "1"
assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY not found — add it to the root .env or run `ant auth login`."
)
print("Credential loaded, color disabled — ant is ready")

## 2. Create an Environment

The environment is the **sandboxed container** where tools run. Create it once; in production you'd persist the id and reuse it.

In [ ]:
%%writefile env.yaml
name: multiturn-env-cli
config:
  type: cloud
  networking:
    type: unrestricted

In [ ]:
# Pipe the YAML in via stdin; capture the new environment's id.
_env = !ant beta:environments create --transform id -r < env.yaml  # type: ignore
env_id = _env[0]
print("Environment ID:", env_id)

## 3. Create an Agent

The agent is a **persisted, versioned config**. The system prompt nudges it to build on the conversation — exactly what multi-turn is about.

In [ ]:
%%writefile agent.yaml
name: Multi-Turn Agent (CLI)
model: claude-opus-4-7
system: |
  You are a helpful assistant. Keep your answers concise and build on the
  conversation so far.
tools:
  - type: agent_toolset_20260401
    default_config:
      enabled: true

In [ ]:
import json

# --format json prints the full object; capture id + version for explicit pinning.
_agent = !ant beta:agents create --format json < agent.yaml  # type: ignore
agent = json.loads("".join(_agent))
agent_id, agent_version = agent["id"], agent["version"]
print("Agent ID :", agent_id)
print("Version  :", agent_version)

## 4. Create ONE Session — Reused Across Every Turn

This is the crux of multi-turn. In module 01 the session was per-run. Here we create **a single session** and send it multiple messages. The server keeps the history, so turn 2 already knows what happened in turn 1.

We keep the session id in `session_id`; the `chat()` helper below reuses it on every turn.

In [ ]:
import json

# Pin the agent to its exact version: {"type": "agent", "id": ..., "version": ...}
agent_ref = json.dumps({"type": "agent", "id": agent_id, "version": agent_version})

_session = !ant beta:sessions create --agent '{agent_ref}' --environment-id {env_id} --title "Multi-turn CLI session" --transform id -r  # type: ignore
session_id = _session[0]
print("Session ID:", session_id)

## 5. A Reusable `chat()` Helper

Every turn repeats the same **stream-first** choreography: open the event stream, send the user message (which triggers the agent loop), then read events until the session goes idle. We wrap that in a `chat()` function so each turn cell is a single call.

Like module 01, streaming needs to read events **live**, so we drive `ant` from Python with `subprocess` (`--format jsonl` buffers until the stream ends; `--format yaml` flushes per event). The one difference from module 01: `chat()` reuses the same `session_id` on every call, so the server keeps the conversation history — you never resend prior turns.

In [ ]:
import subprocess, json

def chat(message: str) -> None:
    """Send one message to the (single, reused) session and stream the reply.

    Stream-first: open the event stream BEFORE sending. Same parser as module 01
    — ant emits each event's `content:` block before its `type:` line, and
    multi-line replies arrive as a `- text: |-` block scalar. The only difference
    here is that we reuse the module-level `session_id` on every call, so the
    server keeps the full conversation history.
    """
    # 1. Open the stream FIRST (stream-before-send).
    stream = subprocess.Popen(
        ["ant", "beta:sessions:events", "stream", "--session-id", session_id, "--format", "yaml"],
        stdout=subprocess.PIPE, text=True, bufsize=1,
    )
    # 2. Send the user message — this triggers the agent loop.
    subprocess.run(
        ["ant", "beta:sessions:events", "send", "--session-id", session_id],
        input=json.dumps({"events": [{"type": "user.message",
                "content": [{"type": "text", "text": message}]}]}),
        stdout=subprocess.DEVNULL, text=True, check=True,
    )
    # 3. Read events until the session goes idle.
    print("Agent: ", end="", flush=True)
    pending, block, buf, indent0 = None, False, [], None
    for raw in stream.stdout:
        line = raw.rstrip("\n")
        if block:  # collecting a block scalar; blank lines belong to it
            if line.strip() == "":
                buf.append(""); continue
            ind = len(line) - len(line.lstrip())
            if indent0 is None or ind >= indent0:
                indent0 = ind if indent0 is None else indent0
                buf.append(line[indent0:]); continue
            pending, block = "\n".join(buf).rstrip("\n"), False  # dedent → block ends
        if line.strip().startswith("- text:"):
            val = line.split("- text:", 1)[1].strip()
            if val in ("|", "|-", ">", ">-", ""):
                block, buf, indent0 = True, [], None
            else:
                pending = val
        elif line == "type: agent.message":
            if pending is not None:
                print(pending, end="", flush=True)
            pending = None
        elif line == "type: user.message":
            pending = None
        elif line in ("type: session.status_idle", "type: session.status_terminated"):
            break
    stream.terminate()
    print()

## 6. Turn 1 — Start the Conversation

Ask a plain question. Nothing special yet — but note we never mention the session history; the server tracks it.

In [ ]:
chat("What's the largest planet in our solar system?")

## 7. Turn 2 — Context Is Preserved

The follow-up says **"it"** and **"they"** — it only makes sense if the agent remembers turn 1. Same session, no history resent.

In [ ]:
chat("How many moons does it have?")

## 8. Turn 3 — Build Deeper

One more follow-up that leans entirely on the accumulated context.

In [ ]:
chat("Name the three largest ones and one interesting fact about each.")

## Summary

You just ran a full multi-turn conversation from the terminal:

```
ant beta:environments create   →  one environment (reused)
ant beta:agents create          →  one versioned agent (reused)
ant beta:sessions create        →  ONE session, reused across turns
per turn:  stream FIRST → send → read until idle   (chat())
```

The key difference from module 01: **the session is created once and reused.** Conversation history is maintained server-side, so each `chat` call sees every earlier turn without resending anything.

### CLI vs SDK, same lesson

| | CLI (this module) | SDK ([`02-multi-turn/`](../02-multi-turn/)) |
|---|---|---|
| Control plane | `ant beta:* create` (version-controlled YAML) | `client.beta.*.create()` |
| Reusable turn helper | `chat()` driving `ant` via `subprocess` | `chat()` calling the SDK |
| Session lifetime | one `session_id`, many sends | one `session.id`, many sends |
| Stream-first | open `ant ...stream` before send | `with ...events.stream()` before send |
| Idle check | `session.status_idle` | `session.status_idle` |

### Next

Continue to the next module for tool use — giving the agent real capabilities (bash, files, code execution) inside its environment.